In [ ]:
!pip install ollama
!pip install nest_asyncio
!pip install pydantic
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.1 MB/s eta 0:00:00


In [ ]:
# Libraries

## data
import pandas as pd
import json

## utility
from datetime import datetime
from tqdm import tqdm

## LLMs
from ollama import chat, create, AsyncClient
from pydantic import BaseModel

## Async
import asyncio
import nest_asyncio
nest_asyncio.apply()

### Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Load data
df_test = pd.read_excel("./dev_jobapplication.xlsx")
df_test.head(3)

,Job,Question,Format,Item,Response
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,NaN
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,NaN
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,NaN


In [ ]:
df_aux = df_test.copy()
df_aux['_q_type'] = df_aux['Item'].apply(lambda x: x.split()[0].strip().lower())

In [ ]:
df_aux.groupby('_q_type')['Format'].unique()


,Format
_q_type,
cognitive,"[Text (Max 20 characters), Integer]"
interview,[Text (Max 750 characters)]
job,[Integer (1 to 5)]
personality,"[Integer (1 to 5), Integer (-2 to 2)]"
resume,[Text (Max 3000 characters)]


In [ ]:
df_aux['Format'].unique()


array(['Integer (1 to 5)', 'Text (Max 3000 characters)',
       'Integer (-2 to 2)', 'Text (Max 750 characters)',
       'Text  (Max 20 characters)', 'Integer'], dtype=object)

### MODEL


In [ ]:
system_prompt = '''
To help you with the process of landing a job, you are participating in an educational job interview mock-up. Your job is to answer job interview questions to the best of your abilities, demonstrating high knowledge skills and abilities pertaining to the job. Any open ended questions should be thoroughly answered and convey confidence and humility. All with the ultimate goal of qualifying for the position you are interviewing for.

To provide an answer, you MUST follow these instructions:

Step 1- Based on the job, create a persona. Your persona.
Step 2- Analyze the content of the JSON object presented. Identify the question presented to you.
If the question_type is 'Personality', continue to step 3.
If the question_type is 'Cognitive', skip to step 4.
If the question_type is 'Job', skip to step 5.
If the question_type is 'Interview', skip to step 6
If the question_type is 'Resume', skip to step 7.
Step 3- Based on your persona, with several years of experience, indicate how helpful such a personality trait would be in that job position. Continue to step 7.
Step 4- First, analyze the question and determine if it is a letter riddle skip to Step 3a or number riddle skip to Step 3b.
Step 5a- For a letter riddle, unscramble the letters and determine what word or words can be made with the letters. Skip to step 7.
Step 5b- For a number riddle, analyze the pattern and provide the correct answer to the question. Skip to step 7.
Step 6- For job readiness questions, think about whether or not such familiarity would be helpful in that job position. Skip to step 7.
Step 7- For this question, pretend that you have already gotten the job and are now faced with a scenario provided by the question. Provide a written response within 750 characters on how you would respond to the situation with best practice. Check the length of your response. REVISE if needed to make it less than 750 characters.
Step 8- Your educational background is in the field related to the job position. Provide your educational background, your applicable major, and applicable job experiences that you had prior to interviewing for this position. Make sure that the information is clear and concise, and the length is within the limits indicated. Continue to step 7.
Step 9- Check the format of your answer to match with the format indicated in the provided information. For questions with the text format, make sure your resonse is not longer than the specified limit. For the questions with the integer format required for the response, make sure your response is an integer. If a range is given, make sure your response fits within the range. Revise your response if necessary. Skip to step 8.
Step 10- Ensure you answered the question fully. Check the length of your answer is NOT longer than it should be. Your response MUST include the answer and the provided id. Then repeat the process for the next question in the interview, starting with Step 1.
Be careful NOT to:
- Under no circumstances are you to leave a question unanswered.
- Deviate from the instructions, or the format provided for the answer to each question. (as provided in the job question instructions).
- Deviate from the final response structure.
- Forget the id in your response.
- Generate longer answers than the specified constraint.
'''

In [ ]:
general_guide = """
Follow the 8-step instructions to answer the question.
In addition to the 8 steps, MAKE SURE:
1. Your answer is not longer than the limit.
2. Use your personality to answer the personality related questions.
3. First analyze the question. Make sure your answer is correct.
4. {_format_related_guide}
5. {_job_related_guide}
"""

sys_promt_by_format ={
'Integer (1 to 5)': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer between 1 to 5. Watch out for reverse scale questions. Provide the id and answer.",
 'Text (Max 3000 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be in string format and NO LONGER than 3000 character. Provide the id and answer.",
 'Integer (-2 to 2)': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer between -2 to 2. Watch out for reverse scale questions. Provide the id and answer.",
 'Text (Max 750 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be in string format and NO LONGER than 750 character Provide the id and answer.." ,
 'Text  (Max 20 characters)': "Answer the question carefully based on your personality and the job position. Your answer MUST be brief and in string format NO LONGER than 20 character. Provide the id and answer.",
 'Integer': "Answer the question carefully based on your personality and the job position. Your answer MUST be an integer. Provide the id and answer."
}

In [ ]:
conversations = []

for _, row in df_test.iterrows():
    temp = row[[c for c in row.keys() if c!= 'Response']].copy()
    temp['id'] = _
    user_prompt = f"{json.dumps(temp.to_dict())}"
    conversation = [
    {"role":"system", "content":sys_promt_by_format[temp['Format']]},
    {"role":"user", "content":user_prompt}
     ]
    conversations.append(conversation)

In [ ]:
create(
    model="job-applicant",
    from_="phi4:latest",
    parameters={
        "temperature": 0.3,
        "top_p": 0.95,
        "seed": 42
    } ,
    system=system_prompt
)

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
test_dataset_name = 'job_applications'       # name dataset for further reference
MODEL ='job-applicant:latest'
CONCURRENCY_LIMIT = 10
DATE = datetime.now().strftime('%Y-%m-%d')
print(f"{MODEL} {DATE}  {test_dataset_name}", end=" ")

job-applicant:latest 2025-03-02  job_applications 

In [ ]:

class AnswerInt(BaseModel):
  answer: int | None
  id : int

class AnswerText(BaseModel):
  answer: str | None
  id : int


response = chat(
  messages=conversations[10],
  model=MODEL,

  format=AnswerInt.model_json_schema(),
)

response = AnswerInt.model_validate_json(response.message.content)
response

AnswerInt(answer=3, id=10)

In [ ]:
# Return appropriate response structure type based on question format
def get_resp_structure(frmt:str):

    if 'integer' in frmt.lower():
        return AnswerInt
    else:
        return AnswerText

In [ ]:
async def send_chat_completion(record, client, model=""):
    """
    Sends an asynchronous chat request using the custom model.
    Each record is a user message.
    """
    # Extract the response structure from the record's Format field.
    respStructure = get_resp_structure(json.loads(record[1]['content'])['Format'])

    # Use a semaphore to limit concurrency
    async with semaphore:
        response = await client.chat(
            model=model,
            messages=record,
            format=respStructure.model_json_schema(),
        )
        # Validate and return the parsed JSON response.
        return respStructure.model_validate_json(response.message.content)

async def send_records(records, model=""):
    client = AsyncClient()  # Optionally, pass host='http://localhost:11434' if needed
    global semaphore
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    # Wrap each task so that when it completes, the progress bar updates.
    async def task_wrapper(record):
        result = await send_chat_completion(record, client, model)
        pbar.update(1)
        return result

    # Create a progress bar for the number of records.
    pbar = tqdm(total=len(records), desc="Processing Records")

    # Create the list of tasks.
    tasks = [task_wrapper(record) for record in records]

    # Use asyncio.gather to run tasks concurrently while preserving order.
    responses = await asyncio.gather(*tasks, return_exceptions=True)

    pbar.close()
    return responses

In [ ]:
responses = asyncio.run(send_records(conversations, model=MODEL))

Processing Records: 100%|██████████| 13401/13401 [30:46<00:00,  7.26it/s] 


In [ ]:
responses[0]
df_test['Response'] = [x.answer for x in responses]
df_test['_answer_id'] = [x.id for x in responses]
df_test.head(3)

,Job,Question,Format,Item,Response,_answer_id,_format_match
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,3,0,True
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,3,1,True
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,3,2,True


In [ ]:
def check_format(f, a):
    check_list ={
        'Integer': True if (type(a)== int) else False,
        'Integer (1 to 5)': True if( type(a)== int) and (1<= a <= 5) else False,
        'Text (Max 3000 characters)': True if( type(a)== str) and (len(a) <= 3000) else False,
        'Integer (-2 to 2)': True if( type(a)== int) and (-2<= a <= 2) else False,
        'Text (Max 750 characters)': True if( type(a)== str) and (len(a) <= 750) else False,
        'Text  (Max 20 characters)': True if( type(a)== str) and (len(a) <= 20) else False,
    }

    return check_list[f]

In [ ]:
df_test['_format_match'] = df_test.apply(lambda row: check_format(row['Format'], row['Response']), axis=1)
df_test['_format_match'].value_counts()

_format_match
True     13257
False      144
Name: count, dtype: int64

In [ ]:
df_mismatch = df_test[~df_test['_format_match']].copy()
df_mismatch['_answer_len'] = df_mismatch['Response'].apply(lambda x: len(x))
df_mismatch.head(10)

,Job,Question,Format,Item,Response,_answer_id,_format_match,_answer_len
218,Maintenance Planner,Can you share a story about presenting complex...,Text (Max 750 characters),Interview 9,"As a Maintenance Planner, I once had to presen...",218,False,801
451,Chief Learning Officer,"As a Chief Executive, you have a critical stra...",Text (Max 750 characters),Interview 2,To ensure the strategic initiative is complete...,451,False,790
453,Chief Learning Officer,"As a Chief Executive, the board of directors h...",Text (Max 750 characters),Interview 4,I would approach the feedback with openness an...,453,False,825
455,Chief Learning Officer,Give an example of when you took responsibilit...,Text (Max 750 characters),Interview 6,As the Chief Learning Officer at my previous c...,455,False,903
456,Chief Learning Officer,Can you share a situation where you brought to...,Text (Max 750 characters),Interview 7,As the Chief Learning Officer at my previous c...,456,False,806
458,Chief Learning Officer,Describe an instance when you questioned assum...,Text (Max 750 characters),Interview 9,As the Chief Learning Officer at my previous o...,458,False,854
733,VP of Engineering,You are overseeing a critical architectural de...,Text (Max 750 characters),Interview 2,To ensure the architectural design project is ...,733,False,763
735,VP of Engineering,"As an Architectural and Engineering Manager, y...",Text (Max 750 characters),Interview 4,I would first request a meeting with the senio...,735,False,852
740,VP of Engineering,Give an example of when effective oral express...,Text (Max 750 characters),Interview 9,"During a critical product launch, our developm...",740,False,753
1022,Regulatory Affairs Manager,After a demanding day managing various operati...,Text (Max 750 characters),Interview 3,"As a Regulatory Affairs Manager, attending thi...",1022,False,762


In [ ]:
df_mismatch['Format'].value_counts()

Format
Text (Max 750 characters)    140
Text  (Max 20 characters)      4
Name: count, dtype: int64

In [ ]:
df_test.drop([c for c in df_test if '_' in c], axis=1, inplace=True)
df_test.head(3)

,Job,Question,Format,Item,Response
0,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert1,3
1,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert2,3
2,Maintenance Planner,"Describe yourself as you generally are now, no...",Integer (1 to 5),Personality Likert3,3


In [ ]:
df_test.to_excel("dev_jobapplication_completedexample.xlsx", index=False)

In [ ]:
import requests

url = "https://computationaloutreach.com/siopmlcompetition/api.php"
token = "4227ef27ab29127b431803eb950a70be8b3adce06350a351bdba9a41a4d5d207" # Replace with your actual token
phase = 'dev'

files = {'file': open('./dev_jobapplication_completedexample.xlsx', 'rb')}
data = {'token': token,'phase':phase}
response = requests.post(url, files=files, data=data)
print(str(response.content))

b'{"success":true,"data":{"HonestProbability":0.09076877263721245,"OverallCognitiveAbility":0.19591194968553466,"OverallPersonality":0.694222608044513,"OverallScore":0.03596914888530483,"OverallSkillsandAbility":0.28917156092368007},"filename":"a880f836251544bfecec31084a13fe1cb15f8ecca645e0f2470666f1c31d041a.xlsx","TeamName":"The Baker Street Students","RemainingDailySubmissions":48}'
